In [8]:
#Importing the random library for generating random numbers

import random
from collections import defaultdict


#The code requires this seed so every run generates the same inventory.
random.seed(42)

#Invalid curly quotation marks. The malformed category list (and Shampoo’). Missing indentation.The incorrect return variable inventories. The unused num_categories argument. Negative and zero expiration values. Because expired products are removed, active inventory uses random.randint(1, 365). Inconsistent organization by separating generation, filtering, calculations, and printing into functions.
#Used Docstring for this assignment to ensure transfer of thought process transcends the comments. Telling readers and myself what the file does. 

def generate_inventory(num_products_per_category=10):
    """Return reproducible inventory data for SD Retailing's five categories."""
    categories = [
        "Hand soap",
        "Moisturizer",
        "Sunscreen",
        "Body oil",
        "Shampoo",
    ]
    inventory = {}

    for category in categories:
        inventory[category] = []
        for _ in range(num_products_per_category):
            product = {
                "Quantity": random.randint(10, 100),
                "Unit Price": round(random.uniform(5.0, 500.0), 2),
                # Only unexpired products belong in the digital inventory.
                "Expiration Date": random.randint(1, 365),
                "Advanced Ordered": random.randint(0, 8),
            }
            inventory[category].append(product)

    return inventory

#Part A was my best effort but not understanding the thought process was pretty difficult in the beginning. 
#Flattening the list makes the total and expiration more straightforward
def print_part_a(inventory):
    """Verify the generated inventory and print the requested sample data."""
    all_products = [
        product
        for products_in_category in inventory.values()
        for product in products_in_category
    ]

    print("PART A")
    print(f"Total products generated: {len(all_products)}")
    for category, products in inventory.items():
        print(f"Products in {category}: {len(products)}")

    expiration_days = [product["Expiration Date"] for product in all_products]
    print(f"Minimum expiration days: {min(expiration_days)}")
    print(f"Maximum expiration days: {max(expiration_days)}")

    print("\nFirst product in each category:")
    for category, products in inventory.items():
        print(f"{category}: {products[0]}")


def find_qualifying_products(inventory):
    """Return products meeting all three high-impact near-expiration rules."""
    qualifying_products = []

    for category, products in inventory.items():
        for product in products:
            if (
                product["Unit Price"] > 200
                and product["Expiration Date"] < 30
                and product["Quantity"] > product["Advanced Ordered"]
            ):
                # Copy the dictionary so added analysis fields do not alter inventory.
                qualifying_product = product.copy()
                qualifying_product["Product Category"] = category
                qualifying_product["Inventory Value"] = round(
                    product["Quantity"] * product["Unit Price"], 2
                )
                qualifying_products.append(qualifying_product)

    return qualifying_products


def category_with_largest_value(products):
    """Return the category with the largest combined inventory value, or None."""
    if not products:
        return None

    value_by_category = defaultdict(float)
    for product in products:
        value_by_category[product["Product Category"]] += product["Inventory Value"]

    return max(value_by_category, key=value_by_category.get)


def print_product(product, include_normalized=False):
    """Print the fields required for one qualifying product."""
    print(f"  Product Category: {product['Product Category']}")
    print(f"  Quantity: {product['Quantity']}")
    print(f"  Unit Price: ${product['Unit Price']:,.2f}")
    print(f"  Expiration Date: {product['Expiration Date']} days")
    print(f"  Advanced Ordered: {product['Advanced Ordered']}")
    print(f"  Inventory Value: ${product['Inventory Value']:,.2f}")
    if include_normalized:
        print(f"  Normalized Unit Price: {product['Normalized Unit Price']:.4f}")

#Part B - Teamwork based, did this with some classmates 
def print_part_b(qualifying_products):
    """Calculate and print the requested Part B measures."""
    print("\nPART B")
    print("Qualifying products:")
    for number, product in enumerate(qualifying_products, start=1):
        print(f"Product {number}:")
        print_product(product)

    number_qualifying = len(qualifying_products)
    aggregate_value = sum(
        product["Inventory Value"] for product in qualifying_products
    )

    print(f"\nTotal qualifying products: {number_qualifying}")
    print(f"Aggregate inventory value: ${aggregate_value:,.2f}")

    if number_qualifying == 0:
        print("Average Quantity / Average Advanced Ordered: not defined (no products)")
    else:
        average_quantity = sum(
            product["Quantity"] for product in qualifying_products
        ) / number_qualifying
        average_advanced_ordered = sum(
            product["Advanced Ordered"] for product in qualifying_products
        ) / number_qualifying

        if average_advanced_ordered == 0:
            print(
                "Average Quantity / Average Advanced Ordered: "
                "not defined (average Advanced Ordered is zero)"
            )
        else:
            ratio = average_quantity / average_advanced_ordered
            print(f"Average Quantity / Average Advanced Ordered: {ratio:.4f}")

    largest_category = category_with_largest_value(qualifying_products)
    if largest_category is None:
        print("Largest contributing category: none (no qualifying products)")
    else:
        print(f"Largest contributing category: {largest_category}")

#Ai-Assisted Part C
def print_part_c(qualifying_products):
    """Normalize qualifying prices and print products from highest to lowest."""
    print("\nPART C")
    if not qualifying_products:
        print("No qualifying products are available to normalize.")
        return

    highest_price = max(product["Unit Price"] for product in qualifying_products)

    # Add a new field; the original Unit Price remains unchanged.
    for product in qualifying_products:
        product["Normalized Unit Price"] = product["Unit Price"] / highest_price

    most_expensive = max(qualifying_products, key=lambda product: product["Unit Price"])
    least_expensive = min(qualifying_products, key=lambda product: product["Unit Price"])

    print(f"Most expensive original Unit Price: ${most_expensive['Unit Price']:,.2f}")
    print(f"Least expensive original Unit Price: ${least_expensive['Unit Price']:,.2f}")
    print(
        "Least expensive normalized Unit Price: "
        f"{least_expensive['Normalized Unit Price']:.4f}"
    )

    products_by_normalized_price = sorted(
        qualifying_products,
        key=lambda product: product["Normalized Unit Price"],
        reverse=True,
    )
    print("\nQualifying products by normalized Unit Price (high to low):")
    for number, product in enumerate(products_by_normalized_price, start=1):
        print(f"Product {number}:")
        print_product(product, include_normalized=True)

    print(
        "\nA normalized Unit Price close to 1 means that a product's price is "
        "close to the highest price among the qualifying products. A value of "
        "1 identifies a highest-priced qualifying product, while smaller values "
        "show its price as a proportion of that benchmark."
    )

#Ai-Assisted Part D, Ran the prompt telling Codex to act as a python expert in order to explain thought process
def print_part_d(qualifying_products):
    """Test whether removing the costliest qualifying item changes the leader."""
    print("\nPART D")
    if not qualifying_products:
        print("The comparison cannot be made because there are no qualifying products.")
        return

    original_largest_category = category_with_largest_value(qualifying_products)
    most_expensive = max(qualifying_products, key=lambda product: product["Unit Price"])

    # Remove exactly the selected dictionary by identity, even if values are duplicated.
    remaining_products = [
        product for product in qualifying_products if product is not most_expensive
    ]
    new_largest_category = category_with_largest_value(remaining_products)

    print(
        "Removed product: "
        f"{most_expensive['Product Category']}, "
        f"Unit Price ${most_expensive['Unit Price']:,.2f}"
    )
    print(f"Largest category before removal: {original_largest_category}")
    print(
        "Largest category after removal: "
        f"{new_largest_category if new_largest_category is not None else 'none'}"
    )
    changed = original_largest_category != new_largest_category
    print(f"Did the largest category change? {'Yes' if changed else 'No'}")


def main():
    """Generate the data once, then complete Parts A through D in sequence."""
    inventory = generate_inventory()
    print_part_a(inventory)

    qualifying_products = find_qualifying_products(inventory)
    print_part_b(qualifying_products)
    print_part_c(qualifying_products)
    print_part_d(qualifying_products)


if __name__ == "__main__":
    main()


PART A
Total products generated: 50
Products in Hand soap: 10
Products in Moisturizer: 10
Products in Sunscreen: 10
Products in Body oil: 10
Products in Shampoo: 10
Minimum expiration days: 10
Maximum expiration days: 354

First product in each category:
Hand soap: {'Quantity': 91, 'Unit Price': 60.11, 'Expiration Date': 141, 'Advanced Ordered': 3}
Moisturizer: {'Quantity': 78, 'Unit Price': 66.79, 'Expiration Date': 194, 'Advanced Ordered': 1}
Sunscreen: {'Quantity': 38, 'Unit Price': 343.88, 'Expiration Date': 29, 'Advanced Ordered': 3}
Body oil: {'Quantity': 77, 'Unit Price': 129.45, 'Expiration Date': 284, 'Advanced Ordered': 0}
Shampoo: {'Quantity': 17, 'Unit Price': 124.23, 'Expiration Date': 291, 'Advanced Ordered': 1}

PART B
Qualifying products:
Product 1:
  Product Category: Hand soap
  Quantity: 21
  Unit Price: $297.29
  Expiration Date: 17 days
  Advanced Ordered: 0
  Inventory Value: $6,243.09
Product 2:
  Product Category: Hand soap
  Quantity: 54
  Unit Price: $303.84
 